# Chapter 10 — Chatbots และ Retrieval-Augmented Generation (RAG)

เป้าหมายของ Lab นี้ คือ สร้าง **RAG Chatbot** ที่ตอบคำถามจากเอกสารหลักสูตร


## Setup — ติดตั้งไลบรารีที่ต้องใช้

In [ ]:
!apt-get install -y -qq tesseract-ocr tesseract-ocr-tha > /dev/null
!pip uninstall -y -q pillow
!pip install -q --no-cache-dir pillow
!pip install -q langchain langchain-community sentence-transformers faiss-cpu google-genai groq openai pdfplumber pytesseract pypdfium2 pythainlp pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 35.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 100)
print("พร้อมใช้งานแล้ว")


พร้อมใช้งานแล้ว


---
# Lab 1 — Prepare Documents

**เป้าหมาย:** Document → Chunks

ในโลกจริง เอกสารต้นฉบับมักอยู่ในรูป PDF และการดึงข้อความ (Text Extraction) จาก PDF ภาษาไทย
มักมีปัญหาสระ/วรรณยุกต์หายหรือคำติดกัน โค้ดด้านล่างนี้ให้เสมือนว่าเอกสารทีถูกทำความสะอาดแล้ว
(cleaned text) เพื่อโฟกัสที่แนวคิดเรื่อง Chunking โดยตรง

In [ ]:
# ขั้นตอน 1: Load เอกสาร (จำลองว่าดึงข้อความจากไฟล์ PDF/TXT ของหลักสูตรมาแล้ว)

CURRICULUM_TEXT = """# หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสารสนเทศและเครือข่าย (หลักสูตรใหม่ พ.ศ. 2562)

ภาควิชาเทคโนโลยีสารสนเทศ คณะเทคโนโลยีและการจัดการอุตสาหกรรม
มหาวิทยาลัยเทคโนโลยีพระจอมเกล้าพระนครเหนือ วิทยาเขตปราจีนบุรี

## หมวดที่ 1 ข้อมูลทั่วไป

### รหัสและชื่อหลักสูตร
ภาษาไทย: หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสารสนเทศและเครือข่าย
ภาษาอังกฤษ: Bachelor of Engineering Program in Information and Network Engineering

### ชื่อปริญญาและสาขาวิชา
ชื่อเต็ม (ภาษาไทย): วิศวกรรมศาสตรบัณฑิต (วิศวกรรมสารสนเทศและเครือข่าย)
ชื่อย่อ (ภาษาไทย): วศ.บ. (วิศวกรรมสารสนเทศและเครือข่าย)
ชื่อเต็ม (ภาษาอังกฤษ): Bachelor of Engineering (Information and Network Engineering)
ชื่อย่อ (ภาษาอังกฤษ): B.Eng. (Information and Network Engineering)

### จำนวนหน่วยกิตที่เรียนตลอดหลักสูตร
135 หน่วยกิต

### รูปแบบของหลักสูตร
หลักสูตรระดับปริญญาตรี หลักสูตร 4 ปี ที่จัดการเรียนการสอนในรูปแบบสหกิจศึกษาและเสริมทักษะภาษาอังกฤษ
ในระหว่างการศึกษาไม่น้อยกว่าร้อยละ 20 ของจำนวนหน่วยกิตรวมตลอดหลักสูตร เป็นหลักสูตรปริญญาตรีทางวิชาการ
การจัดการเรียนการสอนใช้ภาษาไทยและภาษาอังกฤษ รับนักศึกษาไทยหรือนักศึกษาต่างประเทศที่สามารถใช้ภาษาไทยได้

### อาชีพที่สามารถประกอบได้หลังสำเร็จการศึกษา
1. วิศวกรสารสนเทศ
2. วิศวกรสารสนเทศและเครือข่าย
3. วิศวกรข้อมูล
4. วิศวกรเครือข่าย
5. ผู้ดูแลระบบเครือข่าย
6. นักวิชาการคอมพิวเตอร์
7. นักเขียนโปรแกรม ผู้พัฒนาซอฟต์แวร์ หรือนักพัฒนาเว็บไซต์

### สถานที่จัดการเรียนการสอน
ภาควิชาเทคโนโลยีสารสนเทศ คณะเทคโนโลยีและการจัดการอุตสาหกรรม
มหาวิทยาลัยเทคโนโลยีพระจอมเกล้าพระนครเหนือ วิทยาเขตปราจีนบุรี

### คุณสมบัติของผู้เข้าศึกษา
1. สำเร็จการศึกษาระดับมัธยมศึกษาตอนปลาย (ม.6) ที่ผ่านการเรียนวิชาวิทยาศาสตร์และคณิตศาสตร์
ไม่น้อยกว่า 30 หน่วยกิต หรือโดยความเห็นชอบของภาควิชา
2. สำเร็จการศึกษาระดับประกาศนียบัตรวิชาชีพ (ปวช.) สาขาไฟฟ้า อิเล็กทรอนิกส์ เมคคาทรอนิกส์
คอมพิวเตอร์ เทคโนโลยีสารสนเทศ หรือโดยความเห็นชอบของภาควิชา
3. สำหรับนักศึกษาเทียบโอน สำเร็จการศึกษาระดับประกาศนียบัตรวิชาชีพชั้นสูง (ปวส.) สาขาไฟฟ้า
อิเล็กทรอนิกส์ เมคคาทรอนิกส์ คอมพิวเตอร์ เทคโนโลยีสารสนเทศ หรือโดยความเห็นชอบของภาควิชา
4. มีคุณสมบัติอื่น ๆ ตามระเบียบมหาวิทยาลัยเทคโนโลยีพระจอมเกล้าพระนครเหนือว่าด้วยการศึกษาระดับปริญญาบัณฑิต

### ปัญหาของนักศึกษาแรกเข้าและกลยุทธ์แก้ไข
ปัญหาหลักคือการปรับตัวจากการเรียนในระดับมัธยมศึกษามาเป็นการเรียนรู้ที่แตกต่างจากเดิม
ต้องดูแลตนเองมากขึ้น และนักศึกษาแรกเข้ามีพื้นฐานความรู้ทางคอมพิวเตอร์และภาษาอังกฤษที่แตกต่างกัน
กลยุทธ์แก้ไข ได้แก่ การจัดปฐมนิเทศนักศึกษาใหม่ การมอบหมายอาจารย์ที่ปรึกษาดูแลนักศึกษา
และการสอนเสริมปรับพื้นฐานความรู้ทางวิชาการทั้งด้านคอมพิวเตอร์และภาษาอังกฤษ

## หมวดที่ 2 ข้อมูลเฉพาะของหลักสูตร

### ปรัชญาของหลักสูตร
พัฒนาคน เพื่อพัฒนาวิศวกรสารสนเทศและเครือข่าย ซึ่งจะเน้นการปฏิบัติเพื่อเป็นบัณฑิตที่คิดเป็น ทำเป็น
ที่สอดคล้องกับอัตลักษณ์ของมหาวิทยาลัยและคณะ

### ความสำคัญของหลักสูตร
หลักสูตรนี้มีความสำคัญในการสร้างและพัฒนาบุคลากรทางด้านวิศวกรรมสารสนเทศและเครือข่าย
อันเป็นรากฐานที่สำคัญในการขับเคลื่อนและพัฒนาการเจริญเติบโตของอุตสาหกรรมดิจิทัล

### วัตถุประสงค์ของหลักสูตร
1. เพื่อผลิตบัณฑิตด้านวิศวกรรมสารสนเทศและเครือข่ายที่มีความสามารถ ความชำนาญ
เพื่อตอบสนองความต้องการของภาครัฐบาลและเอกชน
2. เพื่อผลิตบัณฑิตให้มีศักยภาพในการวิจัยและพัฒนางานทางด้านวิศวกรรมสารสนเทศและเครือข่าย
3. เพื่อผลิตบัณฑิตให้มีความรู้ มีคุณธรรม จริยธรรม มีความรับผิดชอบต่อสังคม มีระเบียบวินัย
และจรรยาบรรณในการประกอบวิชาชีพ

### จุดเด่นเฉพาะของหลักสูตร
หลักสูตรนี้เป็นการเรียนการสอนทางด้านวิศวกรรมสารสนเทศและเครือข่าย หลักสูตร 4 ปี
โดยเป็นหลักสูตรประเภทเสริมทักษะภาษาอังกฤษ เนื้อหาวิชาในหลักสูตรร้อยละ 20 จะเป็นการจัดการเรียน
การสอนเป็นภาษาอังกฤษ หลักสูตรแบ่งออกเป็น 2 รูปแบบ คือแบบโครงการปกติ กับแบบโครงการสหกิจศึกษา
นอกจากนี้หลักสูตรยังมีความร่วมมือทางวิชาการกับองค์กรชั้นนำ เช่น CCNA CISCO Academy เป็นต้น

## หมวดที่ 3 ระบบการจัดการศึกษา การดำเนินการ และโครงสร้างของหลักสูตร

### ระบบการจัดการศึกษา
ระบบการศึกษาเป็นแบบทวิภาค โดย 1 ปีการศึกษาแบ่งออกเป็น 2 ภาคการศึกษาปกติ และ 1 ภาคฤดูร้อน
แต่ละภาคการศึกษาปกติมีระยะเวลาศึกษาไม่น้อยกว่า 15 สัปดาห์
ภาคการศึกษาต้น: เดือนมิถุนายน ถึง เดือนตุลาคม
ภาคการศึกษาปลาย: เดือนพฤศจิกายน ถึง เดือนมีนาคม
ภาคการศึกษาฤดูร้อน: เดือนเมษายน ถึง เดือนพฤษภาคม
นักศึกษาโครงการปกติต้องเข้ารับการฝึกงาน 240 ชั่วโมง ในปีที่ 3 ภาคการศึกษาฤดูร้อน

### โครงสร้างหลักสูตร (รวม 135 หน่วยกิต)
1. หมวดวิชาศึกษาทั่วไป 31 หน่วยกิต แบ่งเป็นวิชาบังคับ 22 หน่วยกิต และวิชาเลือก 9 หน่วยกิต
2. หมวดวิชาเฉพาะ 98 หน่วยกิต แบ่งเป็น
   - กลุ่มวิชาแกน 47 หน่วยกิต (วิชาพื้นฐานคณิตศาสตร์และวิทยาศาสตร์ 12 หน่วยกิต
     และวิชาพื้นฐานทางวิศวกรรมศาสตร์ 35 หน่วยกิต)
   - กลุ่มวิชาชีพ 43 หน่วยกิต (วิชาบังคับ 34 หน่วยกิต และวิชาเลือก 9 หน่วยกิต)
   - กลุ่มวิชาฝึกงาน/สหกิจศึกษา 8 หน่วยกิต
3. หมวดวิชาเลือกเสรี 6 หน่วยกิต

### แผนการรับนักศึกษา (5 ปีการศึกษา)
ปีการศึกษา 2562 รับนักศึกษาชั้นปีที่ 1 จำนวน 40 คน
ปีการศึกษา 2563 มีนักศึกษารวม 80 คน (ชั้นปี 1 และ 2 อย่างละ 40 คน)
ปีการศึกษา 2564 มีนักศึกษารวม 120 คน (ชั้นปี 1 ถึง 3 อย่างละ 40 คน)
ปีการศึกษา 2565 มีนักศึกษารวม 160 คน (ชั้นปี 1 ถึง 4 อย่างละ 40 คน) และคาดว่าจะมีบัณฑิตรุ่นแรก 40 คน
ปีการศึกษา 2566 มีนักศึกษารวม 160 คน และคาดว่าจะมีบัณฑิตสำเร็จการศึกษา 40 คน

### แผนการศึกษา ปีที่ 1 ภาคการศึกษาที่ 1
วิชาที่เรียน: คณิตศาสตร์วิศวกรรม 1, วิศวกรรมสารสนเทศและเครือข่ายเบื้องต้น, การเขียนโปรแกรมคอมพิวเตอร์,
ปฏิบัติการการเขียนโปรแกรมคอมพิวเตอร์, วิชาเลือกกลุ่มวิชาภาษา 1, วิชาเลือกกลุ่มกีฬาและนันทนาการ,
วิชาเลือกในหมวดวิชาศึกษาทั่วไป 1 รวม 17 หน่วยกิต

### แผนการศึกษา ปีที่ 1 ภาคการศึกษาที่ 2
วิชาที่เรียน: คณิตศาสตร์วิศวกรรม 2, วิชาเลือกกลุ่มวิทยาศาสตร์และคณิตศาสตร์, อิเล็กทรอนิกส์และดิจิทัล,
ปฏิบัติการอิเล็กทรอนิกส์และดิจิทัล, การสื่อสารข้อมูลและเครือข่ายคอมพิวเตอร์, การแก้ปัญหาและขั้นตอนวิธี,
วิชาเลือกกลุ่มวิชาภาษา 2 รวม 19 หน่วยกิต

### แผนการศึกษา ปีที่ 2 ภาคการศึกษาที่ 1
วิชาที่เรียน: คณิตศาสตร์เต็มหน่วยและการประยุกต์, วิศวกรรมข้อมูล, การเขียนโปรแกรมคอมพิวเตอร์ขั้นสูง,
ปฏิบัติการวิศวกรรมเครือข่าย 1, เครือข่ายขั้นสูงและโปรโตคอล, วิชาเลือกกลุ่มวิชาภาษา 3, วิชาเลือกเสรี 1
รวม 19 หน่วยกิต

### แผนการศึกษา ปีที่ 2 ภาคการศึกษาที่ 2
วิชาที่เรียน: การเขียนแบบวิศวกรรมสารสนเทศและเครือข่าย, ระบบฐานข้อมูล, สถาปัตยกรรมคอมพิวเตอร์และ
ระบบปฏิบัติการ, ปฏิบัติการวิศวกรรมเครือข่าย 2, กระบวนการคิดเชิงออกแบบ, วิชาเลือกกลุ่มวิชาภาษา 4,
วิชาเลือกกลุ่มสังคมศาสตร์และมนุษยศาสตร์ รวม 19 หน่วยกิต

### แผนการศึกษา ปีที่ 3 ภาคการศึกษาที่ 1
วิชาที่เรียน: วิศวกรรมซอฟต์แวร์, สถิติสำหรับวิศวกรและนักวิทยาศาสตร์ข้อมูล, ปฏิบัติการวิศวกรรมเครือข่าย 3,
การออกแบบและการจัดทำเครือข่ายคอมพิวเตอร์, การออกแบบและการประยุกต์อินเทอร์เน็ตในทุกสรรพสิ่ง,
วิชาเลือกกลุ่มวิชาชีพ 1, วิชาเลือกในหมวดวิชาศึกษาทั่วไป 2 รวม 19 หน่วยกิต

### แผนการศึกษา ปีที่ 3 ภาคการศึกษาที่ 2 (โครงการปกติ)
วิชาที่เรียน: การจัดการเครือข่ายเบื้องต้นและเครื่องมือ, การประยุกต์ใช้การเรียนรู้ของเครื่องจักร,
ความมั่นคงปลอดภัยไซเบอร์, การพัฒนาโปรแกรมประยุกต์บนอุปกรณ์เคลื่อนที่, หลักการโทรคมนาคม,
วิชาเลือกกลุ่มวิชาชีพ 2 รวม 18 หน่วยกิต

### แผนการศึกษา ปีที่ 3 ภาคการศึกษาที่ 2 (โครงการสหกิจศึกษา)
วิชาที่เรียน: การจัดการเครือข่ายเบื้องต้นและเครื่องมือ, ความมั่นคงปลอดภัยไซเบอร์,
การพัฒนาโปรแกรมประยุกต์บนอุปกรณ์เคลื่อนที่, หลักการโทรคมนาคม, การประยุกต์ใช้การเรียนรู้ของเครื่องจักร,
เตรียมสหกิจศึกษา, วิชาเลือกกลุ่มวิชาชีพ 2 รวม 20 หน่วยกิต

### แผนการศึกษา ปีที่ 3 ภาคฤดูร้อน (โครงการปกติ)
นักศึกษาโครงการปกติต้องลงทะเบียนวิชาการฝึกงาน 2 หน่วยกิต (240 ชั่วโมง) ในภาคฤดูร้อน

### แผนการศึกษา ปีที่ 4 ภาคการศึกษาที่ 1 (โครงการปกติ)
วิชาที่เรียน: สถาปัตยกรรมคลาวด์และการใช้งาน, การวิเคราะห์ข้อมูลขนาดใหญ่,
สัมมนาวิศวกรรมสารสนเทศและเครือข่าย, โครงงานวิศวกรรมสารสนเทศและเครือข่าย 1, วิชาเลือกกลุ่มวิชาชีพ 3,
วิชาเลือกในหมวดวิชาศึกษาทั่วไป 3 รวม 16 หน่วยกิต

### แผนการศึกษา ปีที่ 4 ภาคการศึกษาที่ 2 (โครงการปกติ)
วิชาที่เรียน: โครงงานวิศวกรรมสารสนเทศและเครือข่าย 2, วิชาเลือกเสรี 2 รวม 6 หน่วยกิต

### แผนการศึกษา ปีที่ 4 ภาคการศึกษาที่ 1 (โครงการสหกิจศึกษา)
วิชาที่เรียน: สถาปัตยกรรมคลาวด์และการใช้งาน, การวิเคราะห์ข้อมูลขนาดใหญ่,
สัมมนาวิศวกรรมสารสนเทศและเครือข่าย, วิชาเลือกกลุ่มวิชาชีพ 3, วิชาเลือกในหมวดวิชาศึกษาทั่วไป 3,
วิชาเลือกเสรี 2 รวม 16 หน่วยกิต

### แผนการศึกษา ปีที่ 4 ภาคการศึกษาที่ 2 (โครงการสหกิจศึกษา)
นักศึกษาโครงการสหกิจศึกษาต้องลงทะเบียนวิชาสหกิจศึกษา 6 หน่วยกิต (540 ชั่วโมง) เป็นภาคการศึกษาสุดท้าย

## คำอธิบายรายวิชา (เลือกเฉพาะวิชาสำคัญในสาขา)

### 060233101 วิศวกรรมสารสนเทศและเครือข่ายเบื้องต้น (Introduction to Information and Network Engineering)
3(2-2-5) หน่วยกิต ไม่มีวิชาบังคับก่อน
ระบบคอมพิวเตอร์เบื้องต้น ภาษาคอมพิวเตอร์ ลอจิกที่ใช้ในระบบคอมพิวเตอร์ อุปกรณ์รับ-ส่งข้อมูล
แหล่งข้อมูล การเก็บและดูแลข้อมูล โมเดลข้อมูลเบื้องต้น การรวบรวมข้อมูลและความสามารถในการนำเสนอ
ข้อมูลเบื้องต้น การออกแบบและการวิเคราะห์ข้อมูลขนาดใหญ่เบื้องต้น การสื่อสารข้อมูลและระบบประมวลผล
บนกลุ่มเมฆเบื้องต้น ระบบปฏิบัติการเบื้องต้น แนวโน้มของเทคโนโลยีคอมพิวเตอร์ในอนาคต

### 060233104 การเขียนโปรแกรมคอมพิวเตอร์ (Computer Programming)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
การแทนข้อมูลในคอมพิวเตอร์ การออกแบบและพัฒนาโปรแกรม ขั้นตอนวิธีในการแก้ปัญหา ชนิดข้อมูล
การแทนข้อมูล ตัวแปร นิพจน์ ข้อความสั่งการรับข้อมูลเข้าและส่งข้อมูลออก ข้อความสั่งการตัดสินใจ
การวนซ้ำ แถวลำดับ ฟังก์ชัน โปรแกรมย่อย และการส่งพารามิเตอร์

### 060233107 ระบบฐานข้อมูล (Database System)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
หลักการของระบบฐานข้อมูล ระบบจัดการฐานข้อมูล สถาปัตยกรรมระบบฐานข้อมูล การออกแบบฐานข้อมูล
ภาษาการสืบค้นเชิงโครงสร้าง ความคงสภาพข้อมูล การควบคุมสภาวะการใช้งานพร้อมกัน การจัดการความ
ปลอดภัย การสำรองข้อมูล การคืนสภาพฐานข้อมูล เทคนิคการออกแบบและพัฒนาฐานข้อมูล

### 060233108 การสื่อสารข้อมูลและเครือข่ายคอมพิวเตอร์ (Data Communication and Computer Network)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
องค์ประกอบของการติดต่อสื่อสาร มาตรฐานระบบเปิด รูปแบบของการสื่อสารข้อมูล ลักษณะของสัญญาณ
การส่งสัญญาณ การเข้ารหัส การตรวจสอบและแก้ไขข้อผิดพลาดในระดับเชื่อมโยงข้อมูล เทคโนโลยีของ
เครือข่ายท้องถิ่น มาตรฐานอีเทอร์เน็ต ระบบเครือข่ายไร้สาย ข้อกำหนดไอพีแอดเดรสทั้งเวอร์ชั่นสี่และหก
พื้นฐานการทำงานของชุดโปรโตคอล TCP/IP และการบริการบนอินเทอร์เน็ต

### 060233109 วิศวกรรมซอฟต์แวร์ (Software Engineering)
3(2-2-5) หน่วยกิต ไม่มีวิชาบังคับก่อน
หลักการของวิศวกรรมซอฟต์แวร์ การระบุข้อกำหนดของซอฟต์แวร์ กระบวนการทางซอฟต์แวร์
แบบจำลองซอฟต์แวร์ การประมาณต้นทุนซอฟต์แวร์ การจัดการโครงการ การทดสอบซอฟต์แวร์
คุณภาพซอฟต์แวร์ การจัดทำคู่มือ การส่งมอบซอฟต์แวร์ การบำรุงรักษาซอฟต์แวร์

### 060233112 วิศวกรรมข้อมูล (Data Engineering)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
ภาพรวมของวงชีวิตของข้อมูล วิศวกรรมข้อมูล เทคนิคการออกแบบและโมเดล คลังข้อมูลและการจัดเก็บ
การเข้าถึงและการบำรุงรักษา แอปพลิเคชันและอัลกอริทึมของการวิเคราะห์ข้อมูล วิศวกรรมของข้อมูลที่ไม่ใช่
รูปแบบปกติ มาตรฐานและคุณภาพของข้อมูล

### 060233207 การประยุกต์ใช้การเรียนรู้ของเครื่องจักร (Applied Machine Learning)
3(2-2-5) หน่วยกิต ไม่มีวิชาบังคับก่อน
ความรู้เบื้องต้นและทฤษฎีเกี่ยวกับเทคนิคการเรียนรู้ของเครื่องจักร การเรียนรู้แบบมีผู้สอน เช่น การเรียนรู้
โดยอาศัยพื้นฐานทางสถิติ การเรียนรู้โดยอาศัยการตัดสินใจเชิงต้นไม้ การเรียนรู้โดยอาศัยโครงข่ายประสาทเทียม
การเรียนรู้แบบไม่มีผู้สอน เช่น การจัดกลุ่มข้อมูล การวัดประสิทธิภาพโมเดล และการประยุกต์ใช้ในทางปฏิบัติ

### 060233208 ความมั่นคงปลอดภัยไซเบอร์ (Cyber Security)
3(3-0-6) หน่วยกิต วิชาบังคับก่อน: 060233108 การสื่อสารข้อมูลและเครือข่ายคอมพิวเตอร์
พื้นฐานของระบบเครือข่าย การรักษาความปลอดภัย นโยบายความปลอดภัย กลไกในการรักษาความปลอดภัย
สำหรับอุปกรณ์และโครงสร้าง รูปแบบการโจมตี เทคโนโลยีในการรักษาความปลอดภัยของระบบเครือข่าย
ไฟร์วอลล์ IPS IDS IPSec PKI การเข้ารหัสและการจัดการ จรรยาบรรณการใช้งาน กฎหมายที่เกี่ยวข้องกับ
การให้บริการเครือข่าย การรักษาความเป็นส่วนตัวบนเว็บ อาชญากรรมคอมพิวเตอร์ การป้องกันการรั่วไหลของข้อมูล

### 060233209 การพัฒนาโปรแกรมประยุกต์บนอุปกรณ์เคลื่อนที่ (Mobile Application Development)
3(3-0-6) หน่วยกิต วิชาบังคับก่อน: 060233104 การเขียนโปรแกรมคอมพิวเตอร์
เทคโนโลยีที่ใช้ในการพัฒนาแอปพลิเคชัน การสร้างแอปพลิเคชันบนอุปกรณ์โมบาย การออกแบบส่วนติดต่อ
กับผู้ใช้และประสบการณ์ของผู้ใช้ ระบบปฏิบัติการสำหรับอุปกรณ์ การเชื่อมต่อและส่งผ่านข้อมูลระหว่างอุปกรณ์
ความปลอดภัยของข้อมูล การตรวจสอบความถูกต้องของข้อมูลและการทำงานของโปรแกรม

### 060233211 สถาปัตยกรรมคลาวด์และการใช้งาน (Cloud Architecture and Application)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
แนวคิดพื้นฐานและคุณลักษณะของการประมวลผลคลาวด์ แพลตฟอร์มการให้บริการและโครงสร้างพื้นฐานคลาวด์
การพัฒนาซอฟต์แวร์เพื่อรองรับระบบคลาวด์ การบริหารจัดการคลาวด์ เทคโนโลยีเวอร์ชวลไลเซชั่น
การรักษาความปลอดภัยบนคลาวด์ เครื่องมือและซอฟต์แวร์ เช่น IaaS, PaaS, SaaS

### 060233212 การวิเคราะห์ข้อมูลขนาดใหญ่ (Big Data Analytics)
3(2-2-5) หน่วยกิต ไม่มีวิชาบังคับก่อน
หลักการพื้นฐานของการจัดการ จัดเก็บและวิเคราะห์ข้อมูลขนาดใหญ่ เครื่องมือทางสถิติและการวิเคราะห์
โปรแกรม R และ Python เครื่องมือทางด้านแมชชีนเลิร์นนิ่งอื่น ๆ Spark ML และ Streaming สำหรับการ
วิเคราะห์ข้อมูลแบบเรียลไทม์ การเก็บข้อมูลแบบไม่เป็นโครงสร้าง Neural Network และ TensorFlow

### 060233213 การออกแบบและการประยุกต์อินเทอร์เน็ตในทุกสรรพสิ่ง (Internet of Things Application and Design)
3(3-0-6) หน่วยกิต ไม่มีวิชาบังคับก่อน
หลักการ แนวคิด ทฤษฎีของอินเทอร์เน็ตในทุกสรรพสิ่ง โปรโตคอลที่ใช้ในการติดต่อสื่อสาร เช่น MQTT, M2M, TCP/IP
การประยุกต์ใช้งานไมโครคอนโทรลเลอร์หรือซิงเกิลบอร์ดคอมพิวเตอร์ร่วมกับเครือข่ายเซนเซอร์
เทคโนโลยีไร้สาย เช่น Bluetooth, Wi-Fi, RFID, NB-IoT, LoRa ไปยัง IoT Cloud Service

### 060233214 สัมมนาวิศวกรรมสารสนเทศและเครือข่าย (Information and Network Engineering Seminar)
1(0-3-1) หน่วยกิต วิชาบังคับก่อน: โดยความเห็นชอบของภาควิชา
การจัดสัมมนาทางด้านวิศวกรรมสารสนเทศและเครือข่าย การค้นคว้า ความสามารถในการอ่าน
ความเข้าใจ การบรรยายผลการค้นคว้า การเขียนบทความเพื่อนำเสนอในงานประชุมวิชาการ

### 060233401 โครงงานวิศวกรรมสารสนเทศและเครือข่าย 1 (Information and Network Engineering Project I)
3(0-6-3) หน่วยกิต วิชาบังคับก่อน: โดยความเห็นชอบของภาควิชา
โครงงานที่น่าสนใจทางด้านวิศวกรรมสารสนเทศและเครือข่าย เพื่อฝึกให้นักศึกษาได้ค้นคว้า แก้ปัญหา
และประยุกต์ใช้เทคโนโลยีสารสนเทศกับงานด้านต่าง ๆ นักศึกษาต้องจัดทำรายงานปริญญานิพนธ์พร้อมสอบ

### 060233402 โครงงานวิศวกรรมสารสนเทศและเครือข่าย 2 (Information and Network Engineering Project II)
3(0-6-3) หน่วยกิต วิชาบังคับก่อน: 060233401 โครงงานวิศวกรรมสารสนเทศและเครือข่าย 1
โครงงานที่ให้นักศึกษาได้ฝึกการออกแบบงานทางด้านวิศวกรรมสารสนเทศและเครือข่าย พัฒนาระบบงานใหม่
หรือพัฒนางานต่อจากโครงงานวิศวกรรมสารสนเทศและเครือข่าย 1 นักศึกษาต้องจัดทำรายงานปริญญานิพนธ์พร้อมสอบ

### 060233403 การฝึกงาน (Training)
2 หน่วยกิต (240 ชั่วโมง) วิชาบังคับก่อน: โดยความเห็นชอบของภาควิชา
นักศึกษาต้องฝึกงานภาคฤดูร้อนในบริษัทหรือสถานประกอบการที่เกี่ยวข้องกับสาขาวิชาที่เรียน
จำนวน 240 ชั่วโมงขึ้นไป การประเมินผลเป็นพอใจ (S) หรือไม่พอใจ (U)

### 060233501 เตรียมสหกิจศึกษา (Co-operative Education Preparation)
2(2-0-4) หน่วยกิต ไม่มีวิชาบังคับก่อน
หลักการ แนวคิดและปรัชญาสหกิจศึกษา กระบวนการและระเบียบข้อบังคับที่เกี่ยวข้องกับระบบสหกิจศึกษา
เทคนิคการสมัครงานและการสอบสัมภาษณ์ ความรู้พื้นฐานในการปฏิบัติงานในสถานประกอบการ
การพัฒนาบุคลิกภาพ การเขียนรายงานโครงการ และการนำเสนอผลงานโครงการ

### 060233502 สหกิจศึกษา (Co-operative Education)
6 หน่วยกิต (540 ชั่วโมง) วิชาบังคับก่อน: 060233501 เตรียมสหกิจศึกษา
นักศึกษาต้องเข้าฝึกปฏิบัติงานในสถานประกอบการที่เกี่ยวกับวิชาชีพด้านวิศวกรรมสารสนเทศและเครือข่าย
โดยได้รับความเห็นชอบจากภาควิชา มีระยะเวลาการฝึกปฏิบัติงานไม่น้อยกว่า 18 สัปดาห์ หรือ 540 ชั่วโมง
ในภาคเรียนที่ 2 ชั้นปีที่ 4

## องค์ประกอบเกี่ยวกับประสบการณ์ภาคสนาม
หลักสูตรกำหนดให้นักศึกษาสามารถเลือกจะไปฝึกงานหรือสหกิจศึกษาก็ได้ เพื่อเพิ่มพูนความรู้ประสบการณ์
จากการทำงานจริง โครงการปกติฝึกงานในปีที่ 3 ภาคการศึกษาฤดูร้อน ส่วนโครงการสหกิจศึกษาฝึกในปีที่ 4
ภาคเรียนที่ 2 โดยการฝึกงานจัดเต็มเวลาใน 1 ภาคการศึกษาฤดูร้อน และสหกิจศึกษาจัดเต็มเวลาใน 1 ภาคการศึกษาปกติ

## ข้อกำหนดเกี่ยวกับการทำโครงงานหรืองานวิจัย
ข้อกำหนดในการทำโครงงานต้องเป็นหัวข้อที่เกี่ยวข้องกับการประยุกต์เทคโนโลยีเพื่อธุรกิจ หรือเพื่อการเรียน
การสอน หรือเพื่อทำนุบำรุงศิลปวัฒนธรรม โดยมีจำนวนผู้เข้าร่วมโครงการ 2-3 คน โครงงานดำเนินการใน
ภาคการศึกษาที่ 1-2 ของปีการศึกษาที่ 4 จำนวน 6 หน่วยกิต
"""

# บันทึกเป็นไฟล์ .txt เพื่อจำลอง workflow การโหลดจากไฟล์จริง
with open("curriculum_INE.txt", "w", encoding="utf-8") as f:
    f.write(CURRICULUM_TEXT)

print(f"ความยาวเอกสารทั้งหมด: {len(CURRICULUM_TEXT):,} ตัวอักษร")
print("---- ตัวอย่าง 500 ตัวอักษรแรก ----")
print(CURRICULUM_TEXT[:500])


ความยาวเอกสารทั้งหมด: 15,446 ตัวอักษร
---- ตัวอย่าง 500 ตัวอักษรแรก ----
# หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสารสนเทศและเครือข่าย (หลักสูตรใหม่ พ.ศ. 2562)

ภาควิชาเทคโนโลยีสารสนเทศ คณะเทคโนโลยีและการจัดการอุตสาหกรรม
มหาวิทยาลัยเทคโนโลยีพระจอมเกล้าพระนครเหนือ วิทยาเขตปราจีนบุรี

## หมวดที่ 1 ข้อมูลทั่วไป

### รหัสและชื่อหลักสูตร
ภาษาไทย: หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมสารสนเทศและเครือข่าย
ภาษาอังกฤษ: Bachelor of Engineering Program in Information and Network Engineering

### ชื่อปริญญาและสาขาวิชา
ชื่อเต็ม (ภาษาไทย): วิศวกรรมศาสตรบัณฑิต (วิศวกรรม


### ขั้นตอน 2: Extract Text

เอกสารของเราถูกจัดโครงสร้างด้วย Markdown headers (`##`, `###`) อยู่แล้ว ซึ่งเป็นรูปแบบที่ดี
สำหรับการทำ **Structure-based Chunking** — แบ่งตามโครงสร้างเอกสารจริง แทนการตัดตามจำนวน
ตัวอักษรตายตัว (fixed-size chunking) ซึ่งอาจตัดขาดใจความสำคัญกลางประโยค

ฟังก์ชัน `load_document()` ด้านล่างรองรับทั้งไฟล์ `.txt` และ `.pdf` — ถ้าอยากลองใช้เอกสาร PDF
ของจริงแทน ให้อัปโหลดไฟล์เข้า Colab (แถบด้านซ้าย ไอคอนโฟลเดอร์ → Upload) แล้วเปลี่ยน path
ที่ส่งเข้าฟังก์ชันนี้ได้เลย โดยไม่ต้องแก้โค้ดส่วนอื่น

> ⚠️ **หมายเหตุ:** การดึงข้อความจาก PDF ภาษาไทยของจริง (ที่ไม่ผ่านการทำความสะอาดมาก่อน)
> มักมีปัญหาสระ/วรรณยุกต์หายหรือคำติดกัน ตามที่กล่าวถึงในหัวข้อ Lab 1 ด้านบน สาเหตุมักมาจาก
> การเข้ารหัสฟอนต์ใน PDF เอง (cmap ไม่สมบูรณ์) ไม่ใช่สิ่งที่ regex หรือ spell-checker แก้ได้ตรง ๆ

### ตรวจจับคุณภาพการดึงข้อความ + OCR Fallback

`load_document()` จะคำนวณ **สัดส่วนสระ/วรรณยุกต์ต่อจำนวนตัวอักษรไทยทั้งหมด** (diacritic ratio)
หลังดึงข้อความด้วย pdfplumber — ข้อความไทยปกติมักมีสัดส่วนนี้ประมาณ 0.13-0.20 ถ้าต่ำกว่านั้นมาก
แปลว่าสระ/วรรณยุกต์น่าจะหายไปเยอะระหว่างการดึง ระบบจะเตือนและสลับไปใช้ **OCR** (อ่านจากภาพ
หน้ากระดาษด้วย Tesseract) แทนโดยอัตโนมัติ ซึ่งมักได้ผลลัพธ์ที่สมบูรณ์กว่าเพราะไม่พึ่ง text layer
ของ PDF เลย แลกกับเวลาที่ใช้นานกว่ามาก (ประมาณ 5 วินาที/หน้า)

In [ ]:
import os
import re

DIACRITIC_RATIO_THRESHOLD = 0.12  # ต่ำกว่านี้ถือว่าน่าสงสัยว่าสระ/วรรณยุกต์หายระหว่างดึงข้อความ


def thai_diacritic_ratio(text: str) -> float:
    """สัดส่วนสระ/วรรณยุกต์ (combining marks) ต่อจำนวนตัวอักษรไทยทั้งหมด
    ใช้เป็นตัวชี้วัดคร่าว ๆ ว่าการดึงข้อความสูญเสียสระ/วรรณยุกต์ไปหรือไม่
    """
    thai_chars = re.findall(r"[\u0e01-\u0e5b]", text)
    if not thai_chars:
        return 1.0  # ไม่มีภาษาไทยในข้อความ ไม่ต้องเช็ค
    diacritics = re.findall(r"[\u0e34-\u0e3a\u0e47-\u0e4e]", text)
    return len(diacritics) / len(thai_chars)


def extract_text_via_ocr(file_path: str) -> str:
    """ดึงข้อความด้วย OCR (Tesseract) โดยแปลงแต่ละหน้าเป็นภาพก่อน
    ใช้เมื่อ text layer ของ PDF ดึงข้อความออกมาแล้วสระ/วรรณยุกต์หายเยอะผิดปกติ
    """
    import pytesseract
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(file_path)
    pages_text = []
    for i, page in enumerate(pdf):
        print(f"  OCR หน้า {i + 1}/{len(pdf)}...", end="\r")
        bitmap = page.render(scale=200 / 72)  # ~200 DPI
        pil_image = bitmap.to_pil()
        pages_text.append(pytesseract.image_to_string(pil_image, lang="tha+eng"))
    print()
    return "\n".join(pages_text)


def load_document(file_path: str, ocr_fallback: bool = True) -> str:
    """โหลดข้อความจากไฟล์เอกสาร รองรับทั้ง .txt และ .pdf
    สำหรับ .pdf: ถ้าคุณภาพข้อความที่ดึงได้ต่ำผิดปกติ (สระ/วรรณยุกต์หายเยอะ)
    และ ocr_fallback=True จะสลับไปใช้ OCR แทนโดยอัตโนมัติ
    """
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == ".pdf":
        import pdfplumber
        pages_text = []
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    pages_text.append(page_text)
        text = "\n".join(pages_text)

        ratio = thai_diacritic_ratio(text)
        if ratio < DIACRITIC_RATIO_THRESHOLD:
            print(
                f"⚠️ อัตราส่วนสระ/วรรณยุกต์ต่ำผิดปกติ ({ratio:.1%}) "
                f"— text layer ของ PDF นี้อาจสูญเสียสระ/วรรณยุกต์ไปบางส่วน"
            )
            if ocr_fallback:
                print(f"กำลังสลับไปใช้ OCR แทน (ใช้เวลานานกว่าเดิมมาก ~5 วินาที/หน้า)...")
                text = extract_text_via_ocr(file_path)
            else:
                print("ocr_fallback=False จึงใช้ข้อความจาก text layer ต่อไปตามเดิม")

        return text

    else:
        raise ValueError(f"ไม่รองรับไฟล์นามสกุล '{ext}' — รองรับเฉพาะ .txt และ .pdf เท่านั้น")


#raw_text = load_document("curriculum_INE.txt")
#print(f"จำนวนบรรทัดทั้งหมด: {len(raw_text.splitlines())}")

# ตัวอย่างการใช้กับไฟล์ PDF (ถ้าอัปโหลดไฟล์ของตัวเองแล้ว ให้ลบ # ออกและเปลี่ยนชื่อไฟล์)
raw_text = load_document("มคอ-2_หลักสูตร-INE 4ปี.pdf")


⚠️ อัตราส่วนสระ/วรรณยุกต์ต่ำผิดปกติ (10.9%) — text layer ของ PDF นี้อาจสูญเสียสระ/วรรณยุกต์ไปบางส่วน
กำลังสลับไปใช้ OCR แทน (ใช้เวลานานกว่าเดิมมาก ~5 วินาที/หน้า)...



### ขั้นตอน 3: Chunking แบบ Structure-based

แทนที่จะตัดทุก ๆ N ตัวอักษร เราจะแบ่ง chunk ตามหัวข้อ (`##` และ `###`) ของเอกสาร
เพราะแต่ละหัวข้อ (เช่น 1 รายวิชา, 1 ภาคการศึกษา) มีขอบเขตเนื้อหาที่ชัดเจนอยู่แล้ว
วิธีนี้ทำให้แต่ละ chunk ไม่ปนกันหลายประเด็น และไม่ตัดขาดใจความกลางคัน

> ⚠️ **หมายเหตุ:** วิธีนี้ใช้ได้กับเอกสารที่มี Markdown header (`##`/`###`) ติดมาด้วยเท่านั้น
> ถ้าโหลดจาก PDF/OCR ของจริง ข้อความที่ได้จะ**ไม่มี** `##`/`###` ติดมาเลย (เพราะในกระดาษจริง
> ไม่มีสัญลักษณ์นี้) — `chunk_by_headers()` จึงเช็คให้อัตโนมัติ ถ้าไม่เจอ header เลยจะสลับไปแบ่งตาม
> **ย่อหน้า** (paragraph-based) แทน เพื่อไม่ให้ได้ chunk เดียวทั้งเอกสาร

In [ ]:
def chunk_by_paragraphs(text: str, min_chunk_length: int = 300) -> list[dict]:
    """แบ่ง chunk ตามย่อหน้า (คั่นด้วยบรรทัดว่าง) ใช้เป็นทางเลือกสำรองสำหรับเอกสาร
    ที่ไม่มีโครงสร้าง Markdown header ชัดเจน (เช่น ข้อความดิบจาก PDF/OCR)
    รวมย่อหน้าสั้น ๆ ที่ติดกันเข้าด้วยกัน จนกว่าจะยาวพอ (min_chunk_length) ค่อยตัดเป็น chunk ใหม่
    """
    raw_paragraphs = re.split(r"\n\s*\n", text)
    raw_paragraphs = [p.strip() for p in raw_paragraphs if p.strip()]

    merged = []
    buffer = ""
    for para in raw_paragraphs:
        buffer = f"{buffer}\n\n{para}".strip() if buffer else para
        if len(buffer) >= min_chunk_length:
            merged.append(buffer)
            buffer = ""
    if buffer:
        if merged:
            merged[-1] = f"{merged[-1]}\n\n{buffer}"
        else:
            merged.append(buffer)

    return [
        {"section": "(ไม่พบโครงสร้าง header)", "title": f"ส่วนที่ {i + 1}", "text": c}
        for i, c in enumerate(merged)
    ]


def chunk_by_headers(text: str) -> list[dict]:
    """แบ่งเอกสารเป็น chunk ตามหัวข้อ ### (ระดับย่อยสุด) โดยแนบหัวข้อ ## (หมวดใหญ่) ไว้ด้วย
    เพื่อให้แต่ละ chunk มีบริบทว่าอยู่ในหมวดไหน ถ้าเอกสารไม่มี header เลย (เช่น ข้อความจาก PDF/OCR
    ที่ไม่มี Markdown ติดมา) จะสลับไปใช้ chunk_by_paragraphs() แทนโดยอัตโนมัติ
    """
    lines = text.split("\n")

    has_headers = any(line.startswith("## ") or line.startswith("### ") for line in lines)
    if not has_headers:
        print("ไม่พบ Markdown header (##/###) ในเอกสาร — สลับไปใช้ chunk_by_paragraphs() แทน")
        return chunk_by_paragraphs(text)

    chunks = []
    current_h2 = ""
    current_h3 = ""
    current_body = []

    def flush():
        body = "\n".join(current_body).strip()
        if body:
            header_prefix = f"[{current_h2}] " if current_h2 else ""
            chunks.append({
                "section": current_h2,
                "title": current_h3,
                "text": f"{header_prefix}{current_h3}\n{body}".strip()
            })

    for line in lines:
        if line.startswith("## "):
            flush()
            current_body = []
            current_h2 = line.replace("## ", "").strip()
            current_h3 = ""
        elif line.startswith("### "):
            flush()
            current_body = []
            current_h3 = line.replace("### ", "").strip()
        else:
            current_body.append(line)
    flush()
    return chunks

chunks = chunk_by_headers(raw_text)
print(f"จำนวน Chunk ที่ได้: {len(chunks)}")


ไม่พบ Markdown header (##/###) ในเอกสาร — สลับไปใช้ chunk_by_paragraphs() แทน
จำนวน Chunk ที่ได้: 261


### ขั้นตอน 4: ดู Chunks ที่ได้

แสดงตัวอย่าง chunk แรก ๆ เพื่อตรวจสอบว่าการแบ่งทำงานถูกต้อง

In [ ]:
df_chunks = pd.DataFrame(chunks)
df_chunks["length"] = df_chunks["text"].str.len()
df_chunks.head(10)


,section,title,text,length
0,(ไม่พบโครงสร้าง header),ส่วนที่ 1,หลักสูตรวิศวกรรมศาสตรบัณฑิต\nสาขาวิชาวิศวกรรมสารสนเทศและเครือข่าย\n(หลักสูตรใหม่ พ.ศ. 2562)\n\nภ...,2222
1,(ไม่พบโครงสร้าง header),ส่วนที่ 2,แผนที่แสดงการกระจายความรับผิดชอบผลการเรียนรู้สู่รายวิชา 85\...,312
2,(ไม่พบโครงสร้าง header),ส่วนที่ 3,การพัฒนาคณาจารย์\nการเตรียมการสําหรับอาจารย์ใหม่\nการพัฒนาความรู้และทักษะให้แก่คณาจารย์\n\nการปร...,327
3,(ไม่พบโครงสร้าง header),ส่วนที่ 4,การประเมินผลการดําเนินการตามรายละเอียดหลักสูตร\nการทบทวนผลการประเมินและวางแผนปรับปรุง\n\nแผนภูมิ...,344
4,(ไม่พบโครงสร้าง header),ส่วนที่ 5,มคอ.2\n\nหน้า\n103\n03\n03\n03\n\n104\n04\n04\n\n105\n05\n05\n05\n06\n06\n07\n\n110\n\n21\n\n4 ...,431
5,(ไม่พบโครงสร้าง header),ส่วนที่ 6,หมวดที่ 1 ข้อมูลทั่วไป\n1. รหัสและชื่อหลักสูตร\nภาษาไทย เ: หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิช...,538
6,(ไม่พบโครงสร้าง header),ส่วนที่ 7,4. จํานวนหน่วยกิตที่เรียนตลอดหลักสูตร\n135 หน่วยกิต\n5. รูปแบบของหลักสูตร\n5.1 รูปแบบ\nหลักสูตรร...,441
7,(ไม่พบโครงสร้าง header),ส่วนที่ 8,5.4 การรับเข้าศึกษา\n\nรับนักศึกษาไทย หรือนักศึกษาต่างประเทศที่สามารถใช้ภาษาไทยได้\n5.5 ความร่วม...,301
8,(ไม่พบโครงสร้าง header),ส่วนที่ 9,ค\nก\n\nการประชุ\n- ได้รับความเห็\nรประชุ\n\n_¢\noN\nee\n\nAw\n\nSo\nNO\neo\n\n=\n5\n\n3\n\nประช...,310
9,(ไม่พบโครงสร้าง header),ส่วนที่ 10,1 ๓\n\n- ได้พิจารณากลันกรองโดยคณะกรรมการพิจารณาหลักสูตรระดับปริญญาบัณฑิต\n\nครั้...,334


> 💡 **สังเกต:** คอลัมน์ `length` แสดงความยาว (จำนวนตัวอักษร) ของแต่ละ chunk — จะเห็นว่า
> chunk ที่มาจากคำอธิบายรายวิชามีความยาวใกล้เคียงกัน เพราะแต่ละวิชามีขอบเขตเนื้อหาใกล้เคียงกัน
> ต่างจาก fixed-size chunking ที่จะตัดความยาวเท่ากันทุก chunk โดยไม่สนใจเนื้อหา

In [ ]:
print("การกระจายความยาว Chunk (ตัวอักษร):")
print(df_chunks["length"].describe())


การกระจายความยาว Chunk (ตัวอักษร):
count     261.000000
mean      419.218391
std       205.043321
min       300.000000
25%       322.000000
50%       350.000000
75%       426.000000
max      2222.000000
Name: length, dtype: float64


### Task → Example → Exercise → Answer

**Task:** เปรียบเทียบ Structure-based Chunking (ที่ทำไปแล้ว) กับ Fixed-size Chunking

**Example:** โค้ดด้านล่างสร้าง fixed-size chunk ขนาด 300 ตัวอักษร (ไม่สนใจโครงสร้างเอกสาร)

In [ ]:
def chunk_fixed_size(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

fixed_chunks = chunk_fixed_size(raw_text, chunk_size=300, overlap=50)
print(f"จำนวน fixed-size chunk: {len(fixed_chunks)}")
print("---- ตัวอย่าง chunk ที่ 5 (สังเกตว่าอาจเริ่ม/จบกลางประโยค) ----")
print(fixed_chunks[5])


จำนวน fixed-size chunk: 440
---- ตัวอย่าง chunk ที่ 5 (สังเกตว่าอาจเริ่ม/จบกลางประโยค) ----
หลักสูตรและความเกี่ยวข้องกับพันธกิจ
ของมหาวิทยาลัย
ความสัมพันธ์กับหลักสูตรอื่นที่เปิดสอนในคณะ/ภาควิชาอื่นของมหาวิทยาลัย
หมวดที่ 2 ข้อมูลเฉพาะของหลักสูตร                                10
ปรัชญา ความสําคัญ และวัตถุประสงค์ของหลักสูตร                      10
แผนพัฒนาปรับปรุง                            


**Exercise:** ลองปรับค่า `chunk_size` และ `overlap` ในเซลล์ข้างบนเป็นค่าอื่น (เช่น 150 หรือ 600)
แล้วสังเกตว่า chunk ที่ได้เปลี่ยนไปอย่างไร — โดยเฉพาะที่ตำแหน่งรอยต่อระหว่าง chunk

**Answer (ตัวอย่างข้อสังเกต):**
- `chunk_size` เล็กเกินไป (เช่น 150) → 1 รายวิชาอาจถูกตัดเป็นหลาย chunk ทำให้ความหมายไม่ครบ
- `chunk_size` ใหญ่เกินไป (เช่น 600) → หลายรายวิชาอาจถูกรวมอยู่ใน chunk เดียว ทำให้เวกเตอร์สื่อความหมายไม่ชัดเจน
- นี่คือเหตุผลที่ Lab นี้เลือกใช้ **Structure-based Chunking** แทน เพราะเอกสารมีโครงสร้างชัดเจนอยู่แล้ว

---
# Lab 2 — Embeddings & FAISS

**เป้าหมาย:** เห็นว่า Retrieval เกิดขึ้นได้ **ก่อน** **โดยไม่ต้องมี** LLM เลย

### ขั้นตอน 1: โหลด Embedding Model

ใช้ `paraphrase-multilingual-MiniLM-L12-v2` ซึ่งรองรับภาษาไทยและหลายภาษาใน vector space

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("โหลด Embedding Model สำเร็จ")
print(f"ขนาดเวกเตอร์ (dimension): {embedding_model.get_sentence_embedding_dimension()}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

โหลด Embedding Model สำเร็จ
ขนาดเวกเตอร์ (dimension): 384


/tmp/ipykernel_3950/758869117.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"ขนาดเวกเตอร์ (dimension): {embedding_model.get_sentence_embedding_dimension()}")


### ขั้นตอน 2: สร้าง Embeddings สำหรับทุก Chunk

In [ ]:
chunk_texts = df_chunks["text"].tolist()
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)

print(f"สร้าง Embeddings สำเร็จ: {chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} มิติ")


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

สร้าง Embeddings สำเร็จ: 261 chunks x 384 มิติ


### ขั้นตอน 3: สร้าง FAISS Index

เนื่องจากเรา normalize เวกเตอร์แล้ว (`normalize_embeddings=True`) การใช้ Inner Product (`IndexFlatIP`)
จะเทียบเท่ากับการใช้ Cosine Similarity

In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(chunk_embeddings)

print(f"FAISS Index พร้อมใช้งาน: {faiss_index.ntotal} vectors")


FAISS Index พร้อมใช้งาน: 261 vectors


### ขั้นตอน 4: ทดลอง Semantic Search

ลองถามคำถามและดู Top-3 chunk ที่ระบบค้นพบ — **สังเกตว่ายังไม่มี LLM เข้ามาเกี่ยวข้องเลย
นี่คือขั้นตอน Retrieval ล้วน ๆ**

In [ ]:
def semantic_search(query: str, top_k: int = 3):
    query_vec = embedding_model.encode([query], normalize_embeddings=True)
    scores, indices = faiss_index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": round(float(score), 4),
            "title": df_chunks.iloc[idx]["title"],
            "text": df_chunks.iloc[idx]["text"][:200] + "..."
        })
    return pd.DataFrame(results)

# คำถามตัวอย่าง
query = "นักศึกษาต้องมีคุณสมบัติอะไรจึงจะเข้าเรียนหลักสูตรนี้ได้?"
semantic_search(query, top_k=3)


,score,title,text
0,0.7275,ส่วนที่ 34,1.4 จุดเด่นเฉพาะของหลักสูตร\nหลักสูตรนี่เป็นการเรียนการสอ\n\nหลักสูตรประเภทเสริมทักษะภาษาอังกฤษ\...
1,0.7150,ส่วนที่ 47,5)\n\na\nอ\nค\n\noy\n\n-\nad\nบัณฑิต\n\n๓ๆ\n\nตา\n\nระเ\n\nงนักศึกษาแรกเข้า\nการปรับตัวจากกา\n\n...
2,0.7043,ส่วนที่ 258,19\n\n7 Gur\n\n68 มคอ.2\n\n4. องค์ประ...


**Exercise:** ลองเปลี่ยน `query` เป็นคำถามอื่น เช่น:
- "จบแล้วทำงานอะไรได้บ้าง"
- "วิชา Cyber Security เรียนเกี่ยวกับอะไร"
- "หลักสูตรนี้กี่หน่วยกิต"

แล้วสังเกตค่า `score` (ความคล้าย) ของ chunk อันดับ 1 เทียบกับอันดับ 3 — ยิ่งค่าต่างกันมาก
ยิ่งแปลว่าระบบมั่นใจในคำตอบอันดับ 1 มากขึ้น

In [ ]:
# ลองคำถามอื่น
semantic_search("จบแล้วทำงานอะไรได้บ้าง", top_k=3)


,score,title,text
0,0.3738,ส่วนที่ 39,11\n\nมคอ.2\n\nปีการศึกษาที\n\nความคาดห\n\nวังของผลลัพธ์การเรีย\n\nนรู้ เมือสินปีการศึกษา\n\n3. ...
1,0.3496,ส่วนที่ 225,(Psychology for Work)\n\nวิชาบังคับก่อน : ไม่มี\n\nPrerequisite : None\n\nจิตวิทยาเพื่อการทํางาน...
2,0.2799,ส่วนที่ 223,various economic situations.\n\n080203907ธุรกิจกับชีวิตประจําวัน ...


---
# Lab 3 — Build RAG with Gemini, Groq หรือ OpenRouter

**เป้าหมาย:** ประกอบ Retrieval + Generation เข้าด้วยกัน

```
Question → FAISS Retriever → Top-k Context → Prompt Template → LLM API → Answer
```

### ขั้นตอน 1: เลือก LLM Provider และตั้งค่า API Key

Lab นี้รองรับ 3 ทางเลือก ปรับที่ตัวแปร `LLM_PROVIDER` ด้านล่างได้ตามที่มี API Key พร้อมใช้งาน:

| Provider | ต้องผูกบัตรเครดิตไหม | สมัครได้ที่ |
|---|---|---|
| **Gemini** (Google) | บางบัญชีอาจถูกระบบ anti-abuse บล็อกจนกว่าจะผูก billing (ดูหมายเหตุด้านล่าง) | [Google AI Studio](https://aistudio.google.com/apikey) |
| **Groq** (แนะนำ — เร็ว ไม่ต้องผูกบัตรเลย) | ไม่ต้อง | [console.groq.com](https://console.groq.com/keys) |
| **OpenRouter** (ทางเลือกสำรอง — model ให้เลือกหลากหลาย) | ไม่ต้อง | [openrouter.ai/keys](https://openrouter.ai/keys) |

> ⚠️ **หมายเหตุเรื่อง Gemini:** บางบัญชี (โดยเฉพาะ free tier ที่ยังไม่ผูก billing) อาจเจอ error
> `403 PERMISSION_DENIED: Your project has been denied access` ซึ่งเป็นการ flag บัญชีฝั่ง Google เอง
> ไม่ใช่ปัญหาจากโค้ด หากเจอปัญหานี้ให้สลับมาใช้ Groq หรือ OpenRouter แทนได้ทันทีโดยไม่ต้องแก้โค้ดส่วนอื่น

> ⚠️ **หมายเหตุเรื่อง OpenRouter:** โมเดลฟรี (model id ที่ลงท้ายด้วย `:free`) มีโควตาจำกัดกว่า
> Groq (20 คำขอ/นาที, 50-1,000 คำขอ/วัน) และรายชื่อโมเดลฟรีอาจหมุนเวียนเปลี่ยนแปลงได้บ่อย
> เช็ครายชื่อปัจจุบันที่ [openrouter.ai/models](https://openrouter.ai/models?max_price=0)

(ใน Colab แนะนำให้เก็บ API Key ใน Secrets แทนการเขียนตรง ๆ ในโค้ด)

In [ ]:
LLM_PROVIDER = "groq"  # เปลี่ยนเป็น "gemini" หรือ "openrouter" ได้ตามที่มี API Key พร้อมใช้งาน

if LLM_PROVIDER == "gemini":
    from google import genai

    # วิธีที่ 1: ใช้ Colab Secrets (แนะนำ)
    # from google.colab import userdata
    # GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

    # วิธีที่ 2: ใส่ตรง ๆ (สำหรับทดสอบเท่านั้น ไม่แนะนำสำหรับใช้งานจริง)
    GEMINI_API_KEY = "ใส่ Gemini API Key ของคุณตรงนี้"

    if not GEMINI_API_KEY.strip().isascii():
        raise ValueError(
            "GEMINI_API_KEY ยังเป็นข้อความ placeholder อยู่ (มีภาษาไทยปน) "
            "กรุณาไปสมัครและวาง API Key จริงจาก https://aistudio.google.com/apikey ก่อน"
        )

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    GEMINI_MODEL_NAME = "gemini-3.6-flash"  # ปรับตาม model ล่าสุดที่มีสิทธิ์ใช้งานใน Google AI Studio
    # หมายเหตุ: ชื่อ model ของ Gemini เปลี่ยนรุ่นบ่อย ถ้าเจอ error "model not found"
    # ให้เช็ครายชื่อ model ปัจจุบันที่ https://ai.google.dev/gemini-api/docs/models

    print("ตั้งค่า Gemini API สำเร็จ")

elif LLM_PROVIDER == "groq":
    from groq import Groq

    # วิธีที่ 1: ใช้ Colab Secrets (แนะนำ)
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    # วิธีที่ 2: ใส่ตรง ๆ (สำหรับทดสอบเท่านั้น ไม่แนะนำสำหรับใช้งานจริง)
    #GROQ_API_KEY = "ใส่ Groq API Key ของคุณตรงนี้ (สมัครฟรีที่ https://console.groq.com/keys)"

    if not GROQ_API_KEY.strip().isascii():
        raise ValueError(
            "GROQ_API_KEY ยังเป็นข้อความ placeholder อยู่ (มีภาษาไทยปน) "
            "กรุณาไปสมัครและวาง API Key จริงจาก https://console.groq.com/keys ก่อน "
            "(คีย์จริงจะขึ้นต้นด้วย gsk_ และเป็นภาษาอังกฤษล้วน)"
        )

    groq_client = Groq(api_key=GROQ_API_KEY)
    GROQ_MODEL_NAME = "openai/gpt-oss-120b"  # อยู่บน Free tier ไม่ต้องผูกบัตร; ทางเลือกอื่น: "qwen/qwen3.6-27b"
    # หมายเหตุ: Groq เลิกใช้ (deprecate) รุ่นเก่าค่อนข้างบ่อย ถ้าเจอ error "model_not_found"
    # ให้เช็ครายชื่อ model ปัจจุบันที่ https://console.groq.com/docs/models
    # (เช่น llama-3.3-70b-versatile ถูกเลิกใช้ไปแล้วตั้งแต่ 16 ส.ค. 2569)

    print("ตั้งค่า Groq API สำเร็จ")

elif LLM_PROVIDER == "openrouter":
    from openai import OpenAI

    # วิธีที่ 1: ใช้ Colab Secrets (แนะนำ)
    # from google.colab import userdata
    # OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

    # วิธีที่ 2: ใส่ตรง ๆ (สำหรับทดสอบเท่านั้น ไม่แนะนำสำหรับใช้งานจริง)
    OPENROUTER_API_KEY = "ใส่ OpenRouter API Key ของคุณตรงนี้ (สมัครฟรีที่ https://openrouter.ai/keys)"

    if not OPENROUTER_API_KEY.strip().isascii():
        raise ValueError(
            "OPENROUTER_API_KEY ยังเป็นข้อความ placeholder อยู่ (มีภาษาไทยปน) "
            "กรุณาไปสมัครและวาง API Key จริงจาก https://openrouter.ai/keys ก่อน "
            "(คีย์จริงจะขึ้นต้นด้วย sk-or- และเป็นภาษาอังกฤษล้วน)"
        )

    # OpenRouter ใช้ endpoint ที่เข้ากันได้กับ OpenAI SDK โดยตรง แค่เปลี่ยน base_url
    openrouter_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    OPENROUTER_MODEL_NAME = "openai/gpt-oss-120b:free"  # model ฟรีต้องมี ":free" ต่อท้ายเสมอ
    # หมายเหตุ: รายชื่อ model ฟรีของ OpenRouter หมุนเวียนบ่อยกว่าเจ้าอื่น ถ้าเจอ error "model not found"
    # ให้เช็ครายชื่อปัจจุบันที่ https://openrouter.ai/models?max_price=0

    print("ตั้งค่า OpenRouter API สำเร็จ")

else:
    raise ValueError("LLM_PROVIDER ต้องเป็น 'gemini', 'groq' หรือ 'openrouter' เท่านั้น")


ตั้งค่า Groq API สำเร็จ


### ขั้นตอน 2: ฟังก์ชันเรียก LLM แบบรวม (Unified)

เขียนฟังก์ชันกลาง `generate_answer()` ที่สลับไปเรียก Gemini หรือ Groq ตาม `LLM_PROVIDER`
ที่ตั้งไว้ด้านบน ทำให้โค้ดส่วนที่เหลือของ RAG pipeline ไม่ต้องรู้เลยว่าเบื้องหลังใช้ LLM ตัวไหน

In [ ]:
def generate_answer(prompt: str) -> str:
    if LLM_PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL_NAME,
            contents=prompt,
        )
        return response.text

    elif LLM_PROVIDER == "groq":
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    elif LLM_PROVIDER == "openrouter":
        response = openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    else:
        raise ValueError(f"ไม่รู้จัก LLM_PROVIDER: {LLM_PROVIDER}")


### ขั้นตอน 3: สร้าง Retriever Function

ผูก semantic search (จาก Lab 2) เข้ากับฟังก์ชันที่คืนค่า context พร้อมชื่อ chunk (สำหรับใช้ทำ Citation)

In [ ]:
def retrieve_context(query: str, top_k: int = 3):
    query_vec = embedding_model.encode([query], normalize_embeddings=True)
    scores, indices = faiss_index.search(query_vec, top_k)
    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        retrieved.append({
            "title": df_chunks.iloc[idx]["title"],
            "text": df_chunks.iloc[idx]["text"],
            "score": float(score)
        })
    return retrieved


### ขั้นตอน 4: เขียน RAG Prompt Template

กำหนดกฎ 3 ข้อ:
1. ใช้เฉพาะข้อมูลจาก Context ที่กำหนดให้ในการตอบคำถาม
2. หาก Context ไม่มีข้อมูลเพียงพอ ให้ตอบว่าไม่พบข้อมูล
3. พร้อมระบุแหล่งที่มาของข้อมูล (Citation)

In [ ]:
RAG_PROMPT_TEMPLATE = """คุณเป็นผู้ช่วยตอบคำถามเกี่ยวกับหลักสูตรวิศวกรรมสารสนเทศและเครือข่าย
สำหรับนักศึกษาใหม่ที่กำลังจะเข้าเรียน

กฎการตอบคำถาม:
1. ใช้เฉพาะข้อมูลจาก "บริบท" ด้านล่างในการตอบคำถามเท่านั้น ห้ามใช้ความรู้อื่นนอกเหนือจากนี้
2. หากข้อมูลในบริบทไม่เพียงพอที่จะตอบคำถาม ให้ตอบว่า "ไม่พบข้อมูลนี้ในเอกสารหลักสูตร"
3. ทุกครั้งที่ตอบ ให้ระบุแหล่งที่มา (หัวข้อ) ที่ใช้ตอบคำถามไว้ท้ายคำตอบด้วย

บริบท:
{context}

คำถาม: {question}

คำตอบ:"""


def build_rag_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    context_text = "\n\n".join(
        f"[แหล่งที่มา: {c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return RAG_PROMPT_TEMPLATE.format(context=context_text, question=query)


### ขั้นตอน 5: ประกอบเป็น RAG Chatbot ที่สมบูรณ์

เรียกใช้ `generate_answer()` ที่เขียนไว้ในขั้นตอนที่ 2 — โค้ดตรงนี้จึงทำงานได้เหมือนกันไม่ว่า
จะเลือก `LLM_PROVIDER` เป็น Gemini หรือ Groq

In [ ]:
def rag_chatbot(query: str, top_k: int = 3, verbose: bool = True):
    retrieved = retrieve_context(query, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)

    answer = generate_answer(prompt)

    if verbose:
        print(f"❓ คำถาม: {query}\n")
        print(f"📚 Context ที่ค้นพบ ({len(retrieved)} chunks):")
        for c in retrieved:
            print(f"   - {c['title']} (score={c['score']:.3f})")
        print(f"\n💬 คำตอบ:\n{answer}")

    return answer, retrieved


In [ ]:
# ทดสอบคำถามที่มีคำตอบในเอกสาร
_ = rag_chatbot("นักศึกษาต้องมีคุณสมบัติอะไรจึงจะเข้าเรียนหลักสูตรนี้ได้?")


❓ คำถาม: นักศึกษาต้องมีคุณสมบัติอะไรจึงจะเข้าเรียนหลักสูตรนี้ได้?

📚 Context ที่ค้นพบ (3 chunks):
   - ส่วนที่ 34 (score=0.728)
   - ส่วนที่ 47 (score=0.715)
   - ส่วนที่ 258 (score=0.704)

💬 คำตอบ:
นักศึกษาต้องมีคุณสมบัติดังต่อไปนี้ถึงจะสามารถเข้าเรียนหลักสูตรวิศวกรรมสารสนเทศและเครือข่ายได้  

- มีใบรับรองการสำเร็จการศึกษาระดับมัธยมศึกษา  
- มี **ประกาศนียบัตรวิชาชีพ (ปวช.)** ในสาขา **ไฟฟ้า‑อิเล็กทรอนิกส์** หรือ **เทคโนโลยีคอมพิวเตอร์** (หรือสาขาที่เกี่ยวข้อง)  

**แหล่งที่มา:** ส่วนที่ 47 (รายละเอียดเกี่ยวกับการรับนักศึกษาตามประกาศนียบัตรวิชาชีพ)


**Exercise:** ทดสอบคำถามที่**ไม่มี**คำตอบอยู่ในเอกสาร เช่น "หลักสูตรนี้มีค่าเทอมเท่าไหร่"
แล้วสังเกตว่า Gemini ตอบว่า "ไม่พบข้อมูลนี้ในเอกสารหลักสูตร" ตามกฎที่กำหนดไว้หรือไม่

หากโมเดลแต่งคำตอบขึ้นมาเอง (hallucinate) แทนที่จะบอกว่าไม่พบข้อมูล ให้กลับไปพิจารณาว่า
Prompt Template ควรเข้มงวดกว่านี้หรือไม่

In [ ]:
# ทดสอบคำถามที่ไม่มีคำตอบในเอกสาร
_ = rag_chatbot("หลักสูตรนี้มีค่าเทอมเท่าไหร่")


❓ คำถาม: หลักสูตรนี้มีค่าเทอมเท่าไหร่

📚 Context ที่ค้นพบ (3 chunks):
   - ส่วนที่ 34 (score=0.704)
   - ส่วนที่ 92 (score=0.676)
   - ส่วนที่ 50 (score=0.661)

💬 คำตอบ:
ไม่พบข้อมูลนี้ในเอกสารหลักสูตร  
(อ้างอิงจาก ส่วนที่ 34)


---
# Lab 4 — Test & Evaluate


In [ ]:
test_questions = [
    "หลักสูตรนี้เรียนกี่ปี กี่หน่วยกิต",
    "จบแล้วทำงานอะไรได้บ้าง",
    "วิชา Cyber Security เรียนเกี่ยวกับอะไร",
    "ปีที่ 3 ภาคการศึกษาที่ 1 ต้องเรียนวิชาอะไรบ้าง",
    "หลักสูตรนี้มีทุนการศึกษาให้หรือไม่",  # คำถามที่คาดว่าจะไม่มีคำตอบในเอกสาร
]

eval_results = []
for q in test_questions:
    answer, retrieved = rag_chatbot(q, verbose=False)
    eval_results.append({
        "question": q,
        "top_retrieved": retrieved[0]["title"] if retrieved else "-",
        "top_score": round(retrieved[0]["score"], 3) if retrieved else 0,
        "answer": answer[:150] + ("..." if len(answer) > 150 else "")
    })

df_eval = pd.DataFrame(eval_results)
df_eval


,question,top_retrieved,top_score,answer
0,หลักสูตรนี้เรียนกี่ปี กี่หน่วยกิต,ส่วนที่ 34,0.750,หลักสูตรนี้ใช้ระยะเวลา **4 ปี** ในการสำเร็จการศึกษา \n\n*ข้อมูลที่ระบุว่า “หลักสูตร 4 ปี” พบในส...
1,จบแล้วทำงานอะไรได้บ้าง,ส่วนที่ 39,0.374,จบหลักสูตรวิศวกรรมสารสนเทศและเครือข่ายแล้ว สามารถทำงานได้ในหลายบทบาทที่สอดคล้องกับความรู้และทักษ...
2,วิชา Cyber Security เรียนเกี่ยวกับอะไร,ส่วนที่ 162,0.768,วิชา Cyber Security ศึกษาเกี่ยวกับพื้นฐานและการปฏิบัติด้านความปลอดภัยของระบบเครือข่าย โดยรวมถึง ...
3,ปีที่ 3 ภาคการศึกษาที่ 1 ต้องเรียนวิชาอะไรบ้าง,ส่วนที่ 34,0.743,ไม่พบข้อมูลนี้ในเอกสารหลักสูตร【ส่วนที่ 46】
4,หลักสูตรนี้มีทุนการศึกษาให้หรือไม่,ส่วนที่ 35,0.597,"ไม่พบข้อมูลนี้ในเอกสารหลักสูตร \n**(ส่วนที่ 35, ส่วนที่ 50, ส่วนที่ 86)**"


### ให้ LLM ช่วยประเมินแทน (LLM-as-Judge)

ถ้าไม่อยากประเมินด้วยตนเอง สามารถใช้ LLM อีกครั้งมาช่วย "ตัดสิน" ว่า Retrieved Context
เกี่ยวข้องไหม และ Answer ถูกต้องไหม โดยส่งคำถาม + หัวข้อที่ค้นเจอ + คำตอบ ให้ LLM ประเมินแทนคน

> ⚠️ **ข้อควรระวัง:** ผลจาก LLM-as-Judge อาจไม่แม่นเท่าคนตรวจเองในบางเคส (โดยเฉพาะคำถามที่ตีความ
> ได้หลายแบบ) ควรสุ่มตรวจ (spot-check) ผลลัพธ์เทียบกับการอ่านเองบางข้อเสมอ ถ้าต้องการความเข้มงวด
> ระดับ production แนะนำให้ศึกษา framework เฉพาะทาง เช่น **RAGAS**

In [ ]:
import json
import re


def llm_judge(question: str, retrieved_chunks: list[dict], answer: str) -> dict:
    """ใช้ LLM ตัวเดียวกับที่ตั้งค่าไว้ (Gemini/Groq) มาช่วยตัดสิน Relevant/Correct แทนคน"""
    retrieved_titles = "\n".join(f"- {c['title']}" for c in retrieved_chunks)

    judge_prompt = f"""คุณเป็นผู้ตรวจประเมินระบบ RAG อย่างเป็นกลางและเข้มงวด

คำถาม: {question}

หัวข้อเอกสารที่ระบบค้นเจอ (Retrieved):
{retrieved_titles}

คำตอบที่ระบบให้: {answer}

ประเมิน 2 เรื่อง แล้วตอบเป็น JSON เท่านั้น ห้ามมีข้อความอื่นนอกเหนือจาก JSON:
1. "retrieved_relevant": เอกสารที่ค้นเจอเกี่ยวข้องกับคำถามหรือไม่ -> "Relevant" หรือ "Irrelevant"
2. "answer_correct": คำตอบเหมาะสมหรือไม่ (ถ้าบริบทไม่พอและระบบตอบว่า "ไม่พบข้อมูล" อย่างถูกต้อง
   ให้ถือว่า "Correct" ด้วย) -> "Correct" หรือ "Wrong"

ตัวอย่างรูปแบบคำตอบ: {{"retrieved_relevant": "Relevant", "answer_correct": "Correct"}}"""

    raw = generate_answer(judge_prompt)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return {"retrieved_relevant": "ประเมินไม่ได้", "answer_correct": "ประเมินไม่ได้"}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {"retrieved_relevant": "ประเมินไม่ได้", "answer_correct": "ประเมินไม่ได้"}


auto_eval_rows = []
for q in test_questions:
    answer, retrieved = rag_chatbot(q, verbose=False)
    judgement = llm_judge(q, retrieved, answer)
    relevant = judgement.get("retrieved_relevant", "-")
    correct = judgement.get("answer_correct", "-")

    if correct == "Wrong":
        error_type = "Retrieval Error" if relevant == "Irrelevant" else "Generation Error"
    else:
        error_type = "-"

    auto_eval_rows.append({
        "question": q,
        "retrieved_relevant": relevant,
        "answer_correct": correct,
        "error_type": error_type,
    })

auto_eval = pd.DataFrame(auto_eval_rows)
auto_eval


,question,retrieved_relevant,answer_correct,error_type
0,หลักสูตรนี้เรียนกี่ปี กี่หน่วยกิต,Relevant,Wrong,Generation Error
1,จบแล้วทำงานอะไรได้บ้าง,Irrelevant,Correct,-
2,วิชา Cyber Security เรียนเกี่ยวกับอะไร,Relevant,Correct,-
3,ปีที่ 3 ภาคการศึกษาที่ 1 ต้องเรียนวิชาอะไรบ้าง,Relevant,Wrong,Generation Error
4,หลักสูตรนี้มีทุนการศึกษาให้หรือไม่,Irrelevant,Correct,-


### ทดลองปรับปรุง

เลือกปรับ 1 อย่าง (เช่น `top_k` จาก 3 เป็น 5) แล้วรัน 5 คำถามเดิมซ้ำ เปรียบเทียบผลก่อน/หลัง

In [ ]:
# ทดลองปรับ top_k แล้วรันซ้ำ
eval_results_v2 = []
NEW_TOP_K = 5  # ลองเปลี่ยนค่านี้

for q in test_questions:
    answer, retrieved = rag_chatbot(q, top_k=NEW_TOP_K, verbose=False)
    eval_results_v2.append({
        "question": q,
        "top_retrieved": retrieved[0]["title"] if retrieved else "-",
        "top_score": round(retrieved[0]["score"], 3) if retrieved else 0,
        "answer": answer[:150] + ("..." if len(answer) > 150 else "")
    })

df_eval_v2 = pd.DataFrame(eval_results_v2)
df_eval_v2


,question,top_retrieved,top_score,answer
0,หลักสูตรนี้เรียนกี่ปี กี่หน่วยกิต,ส่วนที่ 34,0.750,หลักสูตรนี้ใช้เวลาเรียน **4 ปี** และต้องสะสม **135 หน่วยกิต** ตลอดหลักสูตร\n\n*แหล่งที่มา: ส่วนท...
1,จบแล้วทำงานอะไรได้บ้าง,ส่วนที่ 39,0.374,"ไม่พบข้อมูลนี้ในเอกสารหลักสูตร \n(แหล่งที่มา: ส่วนที่ 39, 219, 223, 225, 241)"
2,วิชา Cyber Security เรียนเกี่ยวกับอะไร,ส่วนที่ 162,0.768,วิชา **Cyber Security** เรียนเกี่ยวกับ \n\n* พื้นฐานของระบบเครือข่ายและการรักษาความปลอดภัย รวมถ...
3,ปีที่ 3 ภาคการศึกษาที่ 1 ต้องเรียนวิชาอะไรบ้าง,ส่วนที่ 34,0.743,ไม่พบข้อมูลนี้ในเอกสารหลักสูตร【ส่วนที่ 86】
4,หลักสูตรนี้มีทุนการศึกษาให้หรือไม่,ส่วนที่ 35,0.597,"ไม่พบข้อมูลนี้ในเอกสารหลักสูตร \n\nแหล่งที่มา: ส่วนที่ 35, ส่วนที่ 50, ส่วนที่ 86, ส่วนที่ 34, ..."
